In [ ]:
import glob
import os
import platform
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp ,reproject_adaptive, reproject_exact
import numpy as np

In [ ]:
# This is some code to ensure I save my data to the proper directory since I use Linux and Mac OS. Modify as needed
os_check = platform.platform(terse=True)[:5]
if os_check == 'macOS':
	preamble = '/Users/path to user home directory/'
	root2 = f'{preamble}/Deconvolution/'
	root1 = f'{root2}Data/PHOTOMETRY/PHOTOM_CATS/'
else:
	preamble = '/home/ path to user home directory/'
	root2 = f'{preamble}/Deconvolution/'
	root1 = f'{root2}Data/PHOTOMETRY/PHOTOM_CATS/'



final_files_bands = f'{root2}final_file_merge_w_psf.csv'

psf_dir = f'{root2}PSFs/' # path directory for native scale PSFs
psf_rescaled dir = f'{root3}PSFs_rescaled/' # path to directory to save the rescaled PSFs 

image_dir = f'{root2}SPT2106/GOGREEN_IMAGES/native/images/' # <--- this is where you have the native images stored
noise_dir = f'{root2}SPT2106/GOGREEN_IMAGES/native/weights/' # <--- this is where you have the corresponding weights files stored

## Here is the main function I use to do the resampling. You will definitely want yo include scaffolding code when you get the the point whre you are looping over multiple files in several clusters

In [ ]:
def resample_fits(input_file, output_file, new_scale_arcsec):
    """
    Resample a FITS image to a new pixel scale.
    
    Parameters:
        input_file (str): Path to input FITS file
        output_file (str): Path for output resampled FITS file
        new_scale_arcsec (float): New pixel scale in arcseconds/pixel
    """
    
    # Load the original data and header
    with fits.open(input_file) as hdul:
        data = hdul[0].data
        header = hdul[0].header
    
    # Get the original WCS information
    wcs_orig = WCS(header)
    
    # Extract pixel scale, supporting both CD and CDELT
    if 'CD1_1' in header and 'CD2_2' in header:
        cdelt1 = header['CD1_1']
        cdelt2 = header['CD2_2']
    elif 'CDELT1' in header and 'CDELT2' in header:
        cdelt1 = header['CDELT1']
        cdelt2 = header['CDELT2']
    else:
        raise ValueError("No valid pixel scale found in header.")
    
    # Compute original pixel scale in arcsec/pixel
    original_scale_arcsec = ((abs(cdelt1) + abs(cdelt2)) / 2) * 3600  # Average if not square
    print(f"Original pixel scale: {original_scale_arcsec:.4f} arcsec/pixel")
    print(f"New pixel scale: {new_scale_arcsec:.4f} arcsec/pixel")
    
    # Compute resampling scale factor
    scale_factor = original_scale_arcsec / new_scale_arcsec
    
    # Determine output shape
    shape_out = (
        int(round(data.shape[0] * scale_factor)),
        int(round(data.shape[1] * scale_factor))
    )
    
    # Create a new header with updated WCS
    new_header = header.copy()
    new_header['CD1_1'] = cdelt1 / scale_factor
    new_header['CD2_2'] = cdelt2 / scale_factor
    
    # Update CRPIX to maintain the image center
    new_header['CRPIX1'] = (shape_out[1] + 1) / 2
    new_header['CRPIX2'] = (shape_out[0] + 1) / 2
    
    # Update other WCS matrix elements if they exist
    for key in ['CD1_2', 'CD2_1']:
        if key in new_header:
            new_header[key] /= scale_factor
    
    # Remove outdated WCS keywords
    for key in ['LONPOLE', 'LATPOLE']:
        if key in new_header:
            del new_header[key]
    
    # Create new WCS
    wcs_new = WCS(new_header)
    
    # Perform exact reprojection to minimize artifacts
    array_new, footprint = reproject_interp(
        (data, wcs_orig), wcs_new, shape_out=shape_out
    )
    
    # Update NAXIS keywords in header
    new_header['NAXIS1'] = array_new.shape[1]
    new_header['NAXIS2'] = array_new.shape[0]
    
    # Save the resampled image
    fits.writeto(output_file, array_new, new_header, overwrite=True)
    print(f"Resampled image saved to {output_file}")


In [ ]:
# original_sale is .2, this code will compute the original scale from the image and print that out for each image

new_scale = .12

In [ ]:
# you will need to resample both the image file and weights file so you can do something like the following
image_file = 'image.fits' 
noise_file = 'weights.fits'

output_image_file = 'output_image.fits' # name and location of the output file
output_noise_file = 'output_noise.fits' # name and location of output noise file

resample_fits(image_file, output_image_file)
resample_fits(noise_file, output_noise_file)
